In [1]:
import os
import shutil
import random
import xml.etree.ElementTree as ET
from tqdm import tqdm
import cv2
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
dataset_path = r"D:\animal_detection\archive\Dataset"

images_train = os.path.join(dataset_path, "images", "train")
images_val = os.path.join(dataset_path, "images", "val")
annotations_path = os.path.join(dataset_path, "PASCAL_annotation")

labels_train = os.path.join(dataset_path, "labels", "train")
labels_val = os.path.join(dataset_path, "labels", "val")

In [3]:
images_test = os.path.join(dataset_path, "images", "test")
labels_test = os.path.join(dataset_path, "labels", "test")

os.makedirs(images_test, exist_ok=True)
os.makedirs(labels_test, exist_ok=True)

In [4]:
val_files = os.listdir(images_val)

print("Total validation images:", len(val_files))

Total validation images: 182


In [5]:
classes = [
    "deer",
    "fox",
    "capybara",
    "dog",
    "armadillo"
]

In [6]:
class_map = {name: i for i, name in enumerate(classes)}
print(class_map)

{'deer': 0, 'fox': 1, 'capybara': 2, 'dog': 3, 'armadillo': 4}


In [7]:
def convert_voc_to_yolo(xml_file, img_w, img_h):

    tree = ET.parse(xml_file)
    root = tree.getroot()

    yolo_annotations = []

    for obj in root.findall("object"):

        class_name = obj.find("name").text

        if class_name not in class_map:
            continue

        class_id = class_map[class_name]

        bbox = obj.find("bndbox")

        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)

        # Convert to YOLO format
        x_center = ((xmin + xmax) / 2) / img_w
        y_center = ((ymin + ymax) / 2) / img_h
        width = (xmax - xmin) / img_w
        height = (ymax - ymin) / img_h

        yolo_annotations.append(
            f"{class_id} {x_center} {y_center} {width} {height}"
        )

    return yolo_annotations

In [8]:
yolo_path = "BRA_YOLO"

folders = [
    "images/train",
    "images/val",
    "images/test",
    "labels/train",
    "labels/val",
    "labels/test"
]

for folder in folders:
    os.makedirs(os.path.join(yolo_path, folder), exist_ok=True)

In [9]:
dataset_path = r"D:\animal_detection\archive\Dataset"

# Images and labels base folders
images_path = os.path.join(dataset_path, "images")
labels_path = os.path.join(dataset_path, "labels")

In [10]:
images_train = os.path.join(images_path, "train")
images_val = os.path.join(images_path, "val")
images_test = os.path.join(images_path, "test")

# list files
train_files = os.listdir(images_train)
val_files = os.listdir(images_val)
test_files = os.listdir(images_test)

# print counts
print("Train images:", len(train_files))
print("Validation images:", len(val_files))
print("Test images:", len(test_files))

Train images: 1460
Validation images: 182
Test images: 181


In [11]:
def process_dataset(file_list, split):

    for file in tqdm(file_list):

        # image path
        img_path = os.path.join(images_path, split, file)

        # xml annotation path
        xml_file = file.replace(".jpg", ".xml")
        xml_path = os.path.join(annotations_path, xml_file)

        img = cv2.imread(img_path)

        if img is None:
            continue

        h, w, _ = img.shape

        yolo_labels = convert_voc_to_yolo(xml_path, w, h)

        # save image
        shutil.copy(
            img_path,
            os.path.join(yolo_path, f"images/{split}", file)
        )

        # save label
        label_file = file.replace(".jpg", ".txt")

        with open(os.path.join(yolo_path, f"labels/{split}", label_file), "w") as f:
            f.write("\n".join(yolo_labels))

In [12]:
yaml_content = f"""
path: {dataset_path}

train: images/train
val: images/val
test: images/test

names:
  0: tapir
  1: jaguarundi
  2: maned_wolf
  3: puma
  4: giant_anteater
"""

with open("bra_dataset.yaml","w") as f:
    f.write(yaml_content)

print(yaml_content)


path: D:\animal_detection\archive\Dataset

train: images/train
val: images/val
test: images/test

names:
  0: tapir
  1: jaguarundi
  2: maned_wolf
  3: puma
  4: giant_anteater



In [13]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="bra_dataset.yaml",
    epochs=10,
    imgsz=640,
    batch=16
)

Ultralytics 8.4.21  Python-3.11.7 torch-2.2.2+cpu CPU (AMD Ryzen 5 5600H with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=bra_dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000194A4001490>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
    

In [14]:
model.val()

Ultralytics 8.4.21  Python-3.11.7 torch-2.2.2+cpu CPU (AMD Ryzen 5 5600H with Radeon Graphics)
Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 700.2390.3 MB/s, size: 114.2 KB)
val: Scanning D:\animal_detection\archive\Dataset\labels\val.cache... 182 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 182/182  0.0s
val: D:\animal_detection\archive\Dataset\images\val\anta00000537.jpg: corrupt JPEG restored and saved
val: D:\animal_detection\archive\Dataset\images\val\anta00000540.jpg: corrupt JPEG restored and saved
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 1.3s/it 15.1s1.6s
                   all        182        200      0.761      0.674      0.771      0.528
                 tapir         39         41      0.794      0.585      0.741      0.503
            jaguarundi         26         31      0.734      0.839      0.801      0.533
 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000194A19A7DD0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
    

In [16]:
results = model.predict(
    source="BRA_YOLO/images/test",
    conf=0.25,
    save=True
)


image 1/181 D:\animal_detection\BRA_YOLO\images\test\anta00000391.jpg: 384x640 1 tapir, 106.8ms
image 2/181 D:\animal_detection\BRA_YOLO\images\test\anta00000396.jpg: 480x640 1 puma, 78.9ms
image 3/181 D:\animal_detection\BRA_YOLO\images\test\anta00000403.jpg: 416x640 3 tapirs, 74.9ms
image 4/181 D:\animal_detection\BRA_YOLO\images\test\anta00000404.jpg: 448x640 1 tapir, 70.7ms
image 5/181 D:\animal_detection\BRA_YOLO\images\test\anta00000408.jpg: 384x640 1 tapir, 55.6ms
image 6/181 D:\animal_detection\BRA_YOLO\images\test\anta00000415.jpg: 448x640 1 tapir, 56.8ms
image 7/181 D:\animal_detection\BRA_YOLO\images\test\anta00000416.jpg: 448x640 1 tapir, 55.4ms
image 8/181 D:\animal_detection\BRA_YOLO\images\test\anta00000419.jpg: 448x640 1 tapir, 57.6ms
image 9/181 D:\animal_detection\BRA_YOLO\images\test\anta00000430.jpg: 352x640 2 tapirs, 1 puma, 65.4ms
image 10/181 D:\animal_detection\BRA_YOLO\images\test\anta00000431.jpg: 480x640 1 tapir, 61.6ms
image 11/181 D:\animal_detection\BRA_Y

In [17]:
import glob

pred_images = glob.glob("runs/detect/predict/*.jpg")

img = cv2.imread(pred_images[0])
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10,10))
plt.imshow(img)
plt.axis("off")

(-0.5, 639.5, 359.5, -0.5)

In [18]:
model.export(format="onnx")

Ultralytics 8.4.21  Python-3.11.7 torch-2.2.2+cpu CPU (AMD Ryzen 5 5600H with Radeon Graphics)

PyTorch: starting from 'D:\animal_detection\runs\detect\train3\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 9, 8400) (5.9 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   --------------------------------------

'D:\\animal_detection\\runs\\detect\\train3\\weights\\best.onnx'

In [20]:
from ultralytics import YOLO
import cv2

model = YOLO(r"D:\animal_detection\runs\detect\train3\weights\best.onnx")

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    results = model(frame)

    annotated = results[0].plot()

    cv2.imshow("Animal Detection", annotated)

    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()

Loading D:\animal_detection\runs\detect\train3\weights\best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.24.3 with CPUExecutionProvider

0: 640x640 (no detections), 54.8ms
Speed: 16.4ms preprocess, 54.8ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 65.6ms
Speed: 2.9ms preprocess, 65.6ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 63.1ms
Speed: 2.9ms preprocess, 63.1ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 63.1ms
Speed: 5.5ms preprocess, 63.1ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 60.5ms
Speed: 4.0ms preprocess, 60.5ms inference, 3.9ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 60.5ms
Speed: 5.7ms preprocess, 60.5ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 60.6ms
Speed: 10.8ms prep